In [1]:
import sys
print(sys.prefix)

In [4]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Check if a GPU is available and set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bigscience/bloomz-7b1")

# Load the model and quantize it to 4-bit
model = AutoModelForCausalLM.from_pretrained("bigscience/bloomz-7b1", 
                                             #device_map='auto', 
                                             load_in_8bit=True,
                                             torch_dtype=torch.float16,
                                             #bnb_4bit_compute_dtype=torch.float16  # Set the compute data type to float16 for better performance
                                            )#.to(device)

# Load the ARCD dataset from CSV
df = pd.read_csv(r'C:\Users\mosab\Desktop\Datasets\General\ARCD\arcd_all.csv', encoding='utf-16', sep='\t')

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


In [ ]:
max_len=0
c=""
q=""
a=""
p=""
for i, row in df:
    
    if((len(row['context'])+len(row['question'])+len(row['answer']))>max_len):
        max_len=len(row['context'])+len(row['question'])+len(row['answer'])
        c=row['context']
        q=row['question']
        a=row['answer']
        
print(max_len+300, "\n", c, "\n", q, "\n", a, "\n", p)

In [9]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np
import re

# Text normalization for comparison (lowercase, remove articles, punctuation, etc.)
def normalize_text(text):
    text = text.lower()
    #text = re.sub(r'\b(a|an|the)\b', ' ', text)  # Remove articles
    text = re.sub(r'[\u064B-\u065F]', '', text)  # Remove diacritics
    text = re.sub(r'\W+', ' ', text)  # Remove punctuation and double whitespaces
    #text = re.sub(r'[^a-zء-ي0-9]+', ' ', text)  # Remove double whitespaces
    #text = ' '.join(text.split())  # Remove extra spaces
    return text

# Exact Match (EM) calculation
def exact_match_score(predicted_answer, actual_answer):
    return int(normalize_text(predicted_answer) == normalize_text(actual_answer))

# F1 Score calculation for a single prediction
def f1_single(predicted_answer, actual_answer):
    # Tokenize predicted and actual answers
    pred_tokens = normalize_text(predicted_answer).split()
    actual_tokens = normalize_text(actual_answer).split()
    
    if (len(pred_tokens) == 0) and (len(actual_tokens) == 0):
        return 1
    elif (len(pred_tokens) == 0):
        return 0
    elif (len(actual_tokens) == 0):
        return 0
     #common_tokens = pred_tokens.intersection(actual_tokens)
    common_tokens = set(pred_tokens) & set(actual_tokens)
    #common_tokens = remove_stopwords_from_set(common_tokens, stopwords_list)
    if len(common_tokens) == 0:
        return 0.0
    
    precision = len(common_tokens) / len(pred_tokens)
    recall = len(common_tokens) / len(actual_tokens)
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

# Function to calculate average EM and F1 scores across the dataset
def evaluate_qa(predicted_answers, actual_answers):
    total_em = 0
    total_f1 = 0
    n = len(predicted_answers)
    
    for predicted, actual in zip(predicted_answers, actual_answers):
        total_em += exact_match_score(predicted, actual)
        total_f1 += f1_single(predicted, actual)
    
    # Calculate the average EM and F1 scores
    em_score = total_em / n
    f1_score_avg = total_f1 / n
    return em_score, f1_score_avg


# Function to compute exact match (EM) and F1 score
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Strip white space in predictions and labels
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    
    exact_match_score, avg_f1_score = evaluate_qa(decoded_preds, decoded_labels)
    
    # Exact Match
    #exact_matches = [1 if pred == label else 0 for pred, label in zip(decoded_preds, decoded_labels)]
    #exact_match_score = sum(exact_matches) / len(exact_matches)
    
    # F1 Score
    #f1_scores = [f1_metric.compute(predictions=[pred], references=[label])['f1'] for pred, label in zip(decoded_preds, decoded_labels)]
    #avg_f1_score = sum(f1_scores) / len(f1_scores)
    
    
    return {
        "exact_match": exact_match_score,
        "f1": avg_f1_score
    }


In [1]:
# max_len=0
# a=""
# for actual_answer in actual_answers:
#     if(len(actual_answer)>max_len):
#         max_len= len(actual_answer)
#         a=actual_answer
        
# print(max_len, "\n", a)

In [10]:
from datetime import datetime
now = datetime.now()
print(now.time())

12:14:35.490126


In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import pandas as pd

# 2. Set up the PEFT configuration (LoRA)
config = LoraConfig(
    r=16,                  # LoRA attention dimension
    lora_alpha=32,          # LoRA scaling factor
    lora_dropout=0.1,       # Dropout rate
    bias="none",            # Optional: Specify which layers to apply LoRA
    task_type="CAUSAL_LM"   # Type of task (causal language modeling)
)

# 3. Apply PEFT to the model
model = get_peft_model(model, config)

# Check model parameters
model.print_trainable_parameters()

# 1. Prepare the dataset (Assumes the CSV has context, title, question, and answers)
df = pd.read_csv(r'C:\Users\mosab\Desktop\Datasets\General\ARCD\arcd_all.csv', encoding='utf-16', sep='\t')
df_train = df[0:int(df.shape[0]*0.8)]
df_val = df[int(df.shape[0]*0.8)+1:int(df.shape[0]*0.9)]
df_test = df[int(df.shape[0]*0.9)+1:df.shape[0]]#-1

# Function to create prompt (ensure it returns a dictionary)
def create_prompt(row):
    title = row['title']
    context = row['context']
    question = row['question']
    context=f"\nTitle: {title}\n"+context
    prompt = f"Context: {context}\n\nQuestion: {question}\nAnswer:"
    return {'input': prompt, 'output': row['answer']}

train_prompts = df_train.apply(create_prompt, axis=1).tolist()
val_prompts = df_val.apply(create_prompt, axis=1).tolist()
test_prompts = df_test.apply(create_prompt, axis=1).tolist()

# Convert to DataFrame and then HuggingFace Dataset
train_df = pd.DataFrame(train_prompts)
val_df = pd.DataFrame(val_prompts)
test_df = pd.DataFrame(test_prompts)

train_hf_dataset = Dataset.from_pandas(train_df)
val_hf_dataset = Dataset.from_pandas(val_df)
test_hf_dataset = Dataset.from_pandas(test_df)

# 5. Tokenization
def preprocess_function(examples):
    inputs = examples['input']
    outputs = examples['output']
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=512)
    labels = tokenizer(outputs, padding="max_length", truncation=True, max_length=512)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Tokenize datasets
train_tokenized = train_hf_dataset.map(preprocess_function, batched=True)
val_tokenized = val_hf_dataset.map(preprocess_function, batched=True)
test_tokenized = test_hf_dataset.map(preprocess_function, batched=True)

# 6. Set up training arguments and train the model
from transformers import Trainer, TrainingArguments
# Optionally, clear cache if running out of memory
import torch
torch.cuda.empty_cache()



# 7. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 8. Fine-tune the model
trainer.train()

# 9. Evaluate the model on the validation set
val_results = trainer.evaluate(eval_dataset=val_tokenized)
print(f"Validation Results: {val_results}")

# 10. Evaluate the model on the test dataset
test_results = trainer.evaluate(eval_dataset=test_tokenized)  # Test dataset evaluation
print(f"Test Results: {test_results}")

trainable params: 7,864,320 || all params: 7,076,880,384 || trainable%: 0.1111


Map:   0%|          | 0/1116 [00:00<?, ? examples/s]

Map:   0%|          | 0/138 [00:00<?, ? examples/s]

Map:   0%|          | 0/139 [00:00<?, ? examples/s]

C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\accelerate\accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss
0,14.109400,13.646808
1,11.531500,10.020324
2,8.593800,8.400101


C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Validation Results: {'eval_loss': 8.400100708007812, 'eval_runtime': 2403.9111, 'eval_samples_per_second': 0.057, 'eval_steps_per_second': 0.029, 'epoch': 2.924731182795699}
Test Results: {'eval_loss': 8.349581718444824, 'eval_runtime': 2421.2439, 'eval_samples_per_second': 0.057, 'eval_steps_per_second': 0.029, 'epoch': 2.924731182795699}


In [12]:
now = datetime.now()
print(now.time())

12:19:14.815110


In [ ]:
model_path = r'C:\Users\mosab\Desktop\Jupyter Notebook\Research\models\'

# Save the entire model (architecture + weights + optimizer state)
model.save(model_path + "bloomz-7b1 - model -  Lara training")
tokenizer.save_pretrained(model_path + "bloomz-7b1 - tokenizer -  Lara training")